In [4]:
# ! uv pip install sqlalchemy psycopg2-binary pyarrow

In [20]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from sqlalchemy import create_engine, inspect
from sqlalchemy.exc import OperationalError

In [9]:
# ---- Paths ----
ROOT_DATA_DIR = Path.cwd().parent.parent / "dataset"
INDIVIDUAL_DATA_DIR = ROOT_DATA_DIR / "which_vlm_data" / "individual_datasets"

In [11]:
# Collect all parquet files (adjust pattern if needed)
PARQUET_FILES = sorted(INDIVIDUAL_DATA_DIR.glob("*.parquet"))
print(f"Found {len(PARQUET_FILES)} parquet files:")
for f in PARQUET_FILES:
    print(" -", f.name)


Found 50 parquet files:
 - ai2d.parquet
 - all_results.parquet
 - aokvqa.parquet
 - chart2text.parquet
 - chartqa.parquet
 - clevr.parquet
 - cocoqa.parquet
 - datikz.parquet
 - diagram_image_to_text.parquet
 - docvqa.parquet
 - dvqa.parquet
 - figureqa.parquet
 - finqa.parquet
 - geomverse.parquet
 - hateful_memes.parquet
 - hitab.parquet
 - iam.parquet
 - iconqa.parquet
 - infographic_vqa.parquet
 - intergps.parquet
 - localized_narratives.parquet
 - mapqa.parquet
 - mimic_cgd.parquet
 - multihiertt.parquet
 - nlvr2.parquet
 - ocrvqa.parquet
 - plotqa.parquet
 - raven.parquet
 - rendered_text.parquet
 - robut_sqa.parquet
 - robut_wikisql.parquet
 - robut_wtq.parquet
 - scienceqa.parquet
 - screen2words.parquet
 - semantic_evaluation.parquet
 - spot_the_diff.parquet
 - st_vqa.parquet
 - tabmwp.parquet
 - tallyqa.parquet
 - tat_qa.parquet
 - textcaps.parquet
 - textvqa.parquet
 - tqa.parquet
 - vistext.parquet
 - visual7w.parquet
 - visualmrc.parquet
 - vqarad.parquet
 - vqav2.parquet


In [21]:


# ---- DB config ----
DB_USER = "vlmrouter"
DB_PASS = "vlmrouter"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "vlmrouter"

# ---- Build SQLAlchemy engine ----
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False,
)

# ---- Test connection ----
def test_connection():
    print("Connecting to PostgreSQL...")

    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT NOW();"))
            row = result.fetchone()
            print("Connection successful!")
            print("Current time on DB:", row[0])
    except OperationalError as e:
        print("❌ Failed to connect to PostgreSQL")
        print(str(e))

if __name__ == "__main__":
    test_connection()


Connecting to PostgreSQL...
❌ Failed to connect to PostgreSQL
(psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Helper to derive table names from filenames

In [19]:
import re

def make_table_name_from_path(path):
    """
    E.g. 'router_train.parquet' -> 'cauldron_router_train'
    """
    stem = path.stem  # 'train', 'val', 'router_train', etc.
    safe = re.sub(r"[^0-9a-zA-Z_]", "_", stem).lower()
    table_name = f"cauldron_{safe}"
    return table_name


### Ingest each split into its own table

In [ ]:
from tqdm import tqdm

for path in tqdm(PARQUET_FILES, desc="Ingesting parquet splits"):
    table_name = make_table_name_from_path(path)
    print(f"\n=== {path.name} -> table '{table_name}' ===")
    
    # Load the parquet as DataFrame
    df = pd.read_parquet(path)
    print(f" - Rows: {len(df)}, Columns: {len(df.columns)}")

    # Does the table already exist?
    exists = inspector.has_table(table_name)

    if not exists:
        print(" - Table does not exist, creating and inserting data.")
        df.to_sql(
            table_name,
            engine,
            index=False,
            if_exists="fail",   # create new table
            chunksize=1000,
        )
    else:
        print(" - Table exists, appending data.")
        df.to_sql(
            table_name,
            engine,
            index=False,
            if_exists="append",  # append to existing table
            chunksize=1000,
        )

print("\n✅ Ingestion complete for all splits.")


In [ ]:
with engine.connect() as conn:
    for path in PARQUET_FILES:
        table_name = make_table_name_from_path(path)
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};"))
        count = result.scalar_one()
        print(f"Table {table_name}: {count} rows")


In [ ]:
split_path = PARQUET_FILES[0]  # or pick by index/name
table_name = make_table_name_from_path(split_path)

df_preview = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 5;", engine)
df_preview
